In [7]:
!pip install gradio -q

from google.colab import drive
drive.mount('/content/drive')
print('Ready ✓')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready ✓


In [3]:
!pip install emoji -q
!pip install ftfy -q
import gradio as gr
import joblib
import numpy as np
import re
import emoji
import ftfy

# Load model and vectorizer from Drive
model      = joblib.load('/content/drive/MyDrive/data/best_model.pkl')
vectorizer = joblib.load('/content/drive/MyDrive/data/tfidf_vectorizer.pkl')

LABEL_NAMES = ['negative', 'neutral', 'positive']

LABEL_CONFIG = {
    'negative': { 'emoji': '😞','color': '#e74c3c', 'label': 'Negative'},
    'neutral':  { 'emoji': '😐', 'color': '#f39c12', 'label': 'Neutral'},
    'positive': { 'emoji': '😊','color': '#2ecc71', 'label': 'Positive'},
}

print('Model loaded ✓')
print(f'Vectorizer vocab size: {len(vectorizer.vocabulary_):,}')

Model loaded ✓
Vectorizer vocab size: 8,562


In [4]:
def preprocess_text(text):
    """Same preprocessing as Notebook 1 pipeline."""
    if not isinstance(text, str) or not text.strip():
        return ""

    # Fix encoding
    text = ftfy.fix_text(text)

    # Convert emoji to text
    text = emoji.demojize(text, delimiters=(" EMOJI_", "_ "))

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove @mentions, keep hashtag words
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)

    # Lowercase (safe for Devanagari)
    text = text.lower()

    # Normalize repeated chars: haaappy → haappy
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Remove special characters (keep Devanagari + Latin + spaces)
    text = re.sub(r'[^a-z0-9\u0900-\u097F\s_]', ' ', text)

    # Collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print('Preprocessing function defined ✓')

Preprocessing function defined ✓


In [5]:
def predict_sentiment(text):
    """
    Takes raw Hinglish text, preprocesses it,
    runs through TF-IDF + SVM, returns prediction.
    """
    if not text or not text.strip():
        return {
            'Prediction': 'Please enter some text',
            'Confidence': '',
            'Label': ''
        }

    # Preprocess
    clean_text = preprocess_text(text)

    if not clean_text:
        return {
            'Prediction': 'Text too short after cleaning',
            'Confidence': '',
            'Label': ''
        }

    # Vectorize
    vec = vectorizer.transform([clean_text])

    # Predict
    pred_label = LABEL_NAMES[model.predict(vec)[0]]

    # Confidence via decision function (SVM)
    try:
        decision = model.decision_function(vec)[0]
        # Convert to pseudo-probabilities via softmax
        exp_d = np.exp(decision - np.max(decision))
        probs = exp_d / exp_d.sum()
        confidence = round(float(np.max(probs)) * 100, 1)
    except:
        confidence = None

    config = LABEL_CONFIG[pred_label]
    emoji_label = f"{config['emoji']} {config['label']}"
    conf_text = f"{confidence}% confident" if confidence else ""

    return {
        emoji_label: confidence / 100 if confidence else 1.0,
    }

print('Prediction function defined ✓')

Prediction function defined ✓


In [6]:
# ── Example texts covering all 3 classes and domains ─────────────────────────
EXAMPLES = [
    ["Yaar aaj ka din bohot achha tha, sach mein amazing feel ho raha hai!"],
    ["Delivery 3 ghante late aayi aur order bhi galat tha. Worst service ever!"],
    ["Aaj office mein meeting thi, phir ghar aa gaya. Normal day tha."],
    ["Naya phone liya, camera quality is absolutely mind-blowing!"],
    ["Exam mein fail ho gaya, itni mehnat bekar gayi. So disappointed."],
    ["Maine chai banayi aur TV dekha, kuch khaas nahi tha aaj."],
    ["Bhai ka result aya first division! Itna proud feel ho raha hoon!"],
    ["Internet service bohot slow hai, kaam karna impossible ho gaya."],
    ["Iss movie ne toh dil jeet liya, absolutely loved every scene!"],
    ["price dekh ke dimag ghoom gaya, itna mehenga product hai yeh?!"],
]

# ── CSS for custom styling ────────────────────────────────────────────────────
custom_css = """
    #title { text-align: center; color: #1F4E79; }
    #subtitle { text-align: center; color: #555; margin-bottom: 10px; }
    #input-box { font-size: 16px; }
    .gradio-container { max-width: 800px; margin: auto; }
    footer { display: none !important; }
"""

# ── Build interface ───────────────────────────────────────────────────────────
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    # Header
    gr.Markdown("# 🇮🇳 Hinglish Sentiment Analyzer", elem_id="title")
    gr.Markdown(
        "**Hindi-English Code-Mixed Sentiment Analysis**  \n"
        "Internship Project · Amity University Punjab, Mohali  \n"
        "Model: Linear SVM + TF-IDF (Unigram + Bigram)",
        elem_id="subtitle"
    )

    gr.Markdown("---")

    with gr.Row():
        with gr.Column(scale=3):
            input_text = gr.Textbox(
                label="Enter Hinglish Text",
                placeholder="Type anything in Hinglish... e.g. Yaar aaj ka din bohot achha tha!",
                lines=4,
                elem_id="input-box"
            )
            with gr.Row():
                submit_btn = gr.Button("Analyze Sentiment", variant="primary", scale=3)
                clear_btn  = gr.Button("Clear", variant="secondary", scale=1)

        with gr.Column(scale=2):
            output_label = gr.Label(
                label="Sentiment Prediction",
                num_top_classes=3
            )

    # Examples section
    gr.Markdown("### Try These Examples:")
    gr.Examples(
        examples=EXAMPLES,
        inputs=input_text,
        label="Click any example to load it"
    )

    # How it works
    gr.Markdown("---")
    with gr.Accordion("How It Works", open=False):
        gr.Markdown("""
        **Pipeline:**
        1. Raw Hinglish text is cleaned (encoding fix, emoji conversion, URL removal)
        2. Text is lowercased (safe for Devanagari script)
        3. TF-IDF vectorizer converts text to 20,000-dimensional feature vector
        4. Linear SVM classifies into Negative / Neutral / Positive
        5. Confidence scores derived from SVM decision function

        **Dataset:** Mendeley Hinglish Sentiment Dataset
        **Best Model:** Linear SVM · Test Accuracy: 74.2% · Test F1: 0.741
        **Classes:** Negative (😞) · Neutral (😐) · Positive (😊)
        """)

    # Model performance
    with gr.Accordion("Model Performance Comparison", open=False):
        gr.Markdown("""
        | Model | Test Accuracy | Test F1 |
        |-------|--------------|---------|
        | **Linear SVM** ⭐ | **74.22%** | **0.7412** |
        | Random Forest | 73.49% | 0.7265 |
        | Logistic Regression | 72.05% | 0.7186 |
        | Naive Bayes (CNB) | 70.12% | 0.7019 |
        | XGBoost | 63.86% | 0.6131 |
        | Naive Bayes (MNB) | 64.34% | 0.6111 |
        | mBERT | 66.51% | 0.6560 |
        | IndicBERT | 54.94% | 0.4801 |
        | XLM-R | 53.98% | 0.4799 |

        *Traditional ML outperforms transformers on this dataset size —
        a known phenomenon in low-resource NLP.*
        """)

    # Wire up buttons
    submit_btn.click(
        fn=predict_sentiment,
        inputs=input_text,
        outputs=output_label
    )
    clear_btn.click(
        fn=lambda: ("", None),
        inputs=None,
        outputs=[input_text, output_label]
    )
    # Also trigger on Enter key
    input_text.submit(
        fn=predict_sentiment,
        inputs=input_text,
        outputs=output_label
    )

print('Interface built ✓')

# ── Launch ────────────────────────────────────────────────────────────────────
demo.launch(
    share=True,       # gives a public URL valid for 72 hours
    debug=False,
    show_error=True
)

/tmp/ipykernel_521/1748155832.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


Interface built ✓
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2dc957349ea5f44f33.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
